# Inference, Evaluation & Dashboard Export

This notebook serves as the final step of the NLP pipeline for the prototype. It does not perform any new model optimization. Instead, it applies the champion model selected in the previous step (XLM-RoBERTa One-vs-Rest with Weighted BCE) to the dataset.

It is the operational follow-up to the final model-selection notebook. Its purpose is not to compare additional models, but to apply the selected production candidate, inspect its behavior visually, and prepare the final prediction output required by the prototype dashboard.

### Workflow:
1. Load the trained model and the hybrid thresholds determined during the K-Fold experiment.
2. Calculate the probabilities (inference) on the review data.
3. Generate scientific evaluations (Precision-Recall curves & Confusion Matrices) for the thesis documentation.
4. Export the final CSV structure with `pred_` columns for direct integration into the Streamlit dashboard.

In [ ]:
import os
import json
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import precision_recall_curve, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Use the available hardware automatically so inference can run on GPU when available and fall back to CPU otherwise.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Configuration and artifact paths

The following configuration points to the model and threshold artifacts exported from Notebook 05. It also defines the subset of labels that are considered stable and useful enough for dashboard integration in the current prototype version.

In [ ]:
# =========================
# CONFIGURATION
# =========================

# Paths to the artifacts from Notebook 05
MODEL_CHECKPOINT_DIR = Path("./output/03_model_training/results_approach_loss_experiment/best_dashboard_model")
THRESHOLDS_PATH = Path("./output/03_model_training/results_approach_loss_experiment/best_dashboard_model/dashboard_thresholds.csv")
DATA_POOL_PATH = "../data/labeling/manual_labeling_pool_hek_viactiv.xlsx"
OVR_MODELS_DIR = MODEL_CHECKPOINT_DIR / "ovr_models"

# Export path
EXPORT_PATH = Path("../data/processed/reviews_with_predictions.csv")
EXPORT_PATH.parent.mkdir(parents=True, exist_ok=True)

# Restrict the production export to the labels that performed well enough to be considered reliable for dashboard use.
ACTIVE_DASHBOARD_LABELS = [
    'auth_registration', 
    'tech_stability_crash', 
    'general_feedback', 
    'document_management', 
    'smarthealth_epa_features'
]

TEXT_COL = "review_text"
ID_COL = "review_id"
TOPIC_COL = "label_topics_raw"

## Load inference data and thresholds

This step loads the review pool that will receive model predictions and imports the calibrated per-label thresholds from the final model-selection stage. These thresholds are important because the dashboard does not rely on a universal 0.5 cutoff, but on label-specific operating points derived from prior evaluation.

In [ ]:
# 1. Load data pool
df_raw = pd.read_excel(DATA_POOL_PATH)
print(f"Dataset loaded: {len(df_raw)} rows.")

# 2. Load thresholds and map them to a dictionary
df_thresholds = pd.read_csv(THRESHOLDS_PATH)
threshold_map = dict(zip(df_thresholds['label'], df_thresholds['threshold']))

print("\nLoaded Dashboard Thresholds:")
for lbl in ACTIVE_DASHBOARD_LABELS:
    print(f" - {lbl}: {threshold_map.get(lbl, 0.50)}")

## Run inference with the selected One-vs-Rest models

Because the final production setup uses a One-vs-Rest formulation, inference is performed separately for each active dashboard label. The resulting probabilities are stored label by label and later combined into one unified prediction table.

In [ ]:
# The tokenizer is identical for all models, load from the first available label
first_model_path = OVR_MODELS_DIR / ACTIVE_DASHBOARD_LABELS[0]
tokenizer = AutoTokenizer.from_pretrained(first_model_path)

texts = df_raw[TEXT_COL].astype(str).tolist()
probs_dict = {}
label_list = ACTIVE_DASHBOARD_LABELS # Define label_list for downstream cells

print("Starting inference on the full data pool (One-vs-Rest)...")

# Disable gradient tracking because this stage performs inference only and does not update model parameters.
with torch.no_grad():
    # Score each dashboard label with its own binary classifier and collect the resulting probabilities.
    for label in ACTIVE_DASHBOARD_LABELS:
        print(f" -> Scoring label: {label}")
        model_path = OVR_MODELS_DIR / label
        
        # Load the specific binary model for this label
        model = AutoModelForSequenceClassification.from_pretrained(model_path).to(device)
        model.eval()
        
        label_probs = []
        # Run inference review by review; this is simple and transparent, even if batching could later improve speed.
        for text in texts:
            inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=256).to(device)
            outputs = model(**inputs)
            prob = torch.sigmoid(outputs.logits).cpu().numpy()[0][0]
            label_probs.append(prob)
                
        probs_dict[label] = np.array(label_probs)
        
        # Remove the current model from memory before loading the next label-specific classifier.
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

print("\nInference completed successfully!")

## Evaluation plots

The following plots are intended primarily for thesis documentation and model interpretation. Precision-recall curves visualize the trade-off between precision and recall across thresholds, while confusion matrices show the concrete classification outcome when the selected dashboard threshold is applied.

In [ ]:
# Create the graph layout for the results chapter of the thesis
fig, axes = plt.subplots(len(ACTIVE_DASHBOARD_LABELS), 2, figsize=(14, 4 * len(ACTIVE_DASHBOARD_LABELS)))
sns.set_theme(style="whitegrid")

for idx, label in enumerate(ACTIVE_DASHBOARD_LABELS):
    # Build ground truth binary array for this specific label
    y_true = np.array([1 if label in [t.strip() for t in str(topics).split(';')] else 0 for topics in df_raw[TOPIC_COL]])
    y_scores = probs_dict[label]
    current_thr = threshold_map.get(label, 0.50)
    
    # --- Left Column: Precision-Recall Curve ---
    precisions, recalls, thrs = precision_recall_curve(y_true, y_scores)
    axes[idx, 0].plot(recalls, precisions, label='PR Curve', color='teal', lw=2)
    
    # Find the point on the curve closest to the selected threshold
    closest_thr_idx = np.argmin(np.abs(thrs - current_thr))
    axes[idx, 0].scatter(recalls[closest_thr_idx], precisions[closest_thr_idx], color='crimson', s=100, zorder=5,
                         label=f'Selected Threshold ({current_thr:.2f})')
    
    axes[idx, 0].set_title(f"Precision-Recall Curve: {label}", fontsize=12, fontweight='bold')
    axes[idx, 0].set_xlabel("Recall")
    axes[idx, 0].set_ylabel("Precision")
    axes[idx, 0].legend(loc="lower left")
    
    # --- Right Column: Confusion Matrix ---
    y_pred = (y_scores >= current_thr).astype(int)
    cm = confusion_matrix(y_true, y_pred)
    
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=axes[idx, 1],
                xticklabels=["Negative", "Positive"], yticklabels=["Negative", "Positive"], annot_kws={"size": 14})
    axes[idx, 1].set_title(f"Confusion Matrix: {label}", fontsize=12, fontweight='bold')
    axes[idx, 1].set_xlabel("Predicted Label")
    axes[idx, 1].set_ylabel("True Label")

plt.tight_layout()
Path("output/03_model_training/results_approach_loss_experiment").mkdir(parents=True, exist_ok=True)
plt.savefig("output/03_model_training/results_approach_loss_experiment/dashboard_evaluation_metrics.png", dpi=300)
plt.show()

## Dashboard export preparation

The final export combines the original review data with one probability column and one binary prediction column per active dashboard label. In addition, a compact summary column is created so the dashboard can directly display the set of detected topics for each review.

In [ ]:
# Create the final export DataFrame
export_df = df_raw.copy()

print("Structuring final prediction columns for the Streamlit dashboard...")
for label in ACTIVE_DASHBOARD_LABELS:
    current_thr = threshold_map.get(label, 0.50)
    
    # Extract probabilities and assign hard binarization (1/0)
    export_df[f"prob_{label}"] = probs_dict[label]
    export_df[f"pred_{label}"] = (probs_dict[label] >= current_thr).astype(int)

# Calculate the column for the UI summary of detected topics per review
final_predicted_topics = []
for i in range(len(export_df)):
    active_topics = [lbl for lbl in ACTIVE_DASHBOARD_LABELS if export_df.loc[i, f"pred_{lbl}"] == 1]
    final_predicted_topics.append(";".join(active_topics))

export_df["predicted_topics_dashboard"] = final_predicted_topics

# Final export to the data directory of the Streamlit app
export_df.to_csv(EXPORT_PATH, index=False, encoding="utf-8")
print(f"\nSuccessfully exported! File ready at:\n{EXPORT_PATH}")

## Final note

This notebook closes the modeling workflow by bridging model selection and prototype integration. It therefore serves two roles at once: documenting the final prediction behavior for the thesis and preparing the processed data structure needed by the Streamlit dashboard.